# Paper 12 W6 — FastSurfer on Google Colab (A100 or T4 GPU)

Runs FastSurfer `--seg_only` on Wang N=161 PPMI T1 cohort (401 scans).
Reads NIfTI tarball + license from Google Drive, writes outputs back to Drive.

**Before running:**
1. Upload `nifti_wang_n161.tar.gz` (4.9 GB) and `license.txt` to `/My Drive/PPMIData_FreeSurfer/`
2. Colab menu → Runtime → Change runtime type → **GPU → A100** (or T4 if A100 unavailable)
3. Runtime → Run all

**Timing:** ~1-2 min/scan on A100 → 401 × 1.5 min ≈ 10 hrs · fits one Colab session (24 hr max).

**Resume:** notebook is idempotent — re-running skips subjects whose `aseg+DKT.stats` exists.

## 1. Mount Drive + verify GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi | head -15

## 2. Unpack NIfTIs + license

In [ ]:
import os, shutil, subprocess
DRIVE_ROOT = '/content/drive/MyDrive/PPMIData_FreeSurfer'
WORK = '/content/work'
os.makedirs(WORK, exist_ok=True)

# Verify Drive files
tar = f'{DRIVE_ROOT}/nifti_wang_n161.tar.gz'
lic = f'{DRIVE_ROOT}/license.txt'
for f in [tar, lic]:
    assert os.path.exists(f), f'MISSING: {f}'
print('files ok')

# Extract (skip if already done)
if not os.path.exists(f'{WORK}/nifti'):
    !mkdir -p {WORK}/nifti && tar -xzf {tar} -C {WORK}/nifti/
    !ls {WORK}/nifti | head
print(f'nifti files: ', end='')
!find {WORK}/nifti -name '*.nii.gz' | wc -l

## 3. Install FastSurfer

In [ ]:
if not os.path.exists('/content/FastSurfer'):
    !git clone --depth 1 --branch stable https://github.com/Deep-MI/FastSurfer.git /content/FastSurfer
!pip install -q -r /content/FastSurfer/requirements.txt 2>&1 | tail -5
!pip install -q 'torch>=2.2' 'torchvision>=0.17' 2>&1 | tail -3
print('install done')

## 4. Sanity test — run on 1 scan

In [ ]:
OUT_DIR = f'{WORK}/fastsurfer_out'
os.makedirs(OUT_DIR, exist_ok=True)

# Find any nifti — may be in nifti/ or nifti/nifti_wang_n161/ depending on tarball structure
import glob
candidates = glob.glob(f'{WORK}/nifti/*.nii.gz') + glob.glob(f'{WORK}/nifti/*/*.nii.gz')
first = sorted(candidates)[0]
sid = os.path.basename(first).replace('.nii.gz', '')
print(f'testing on: {sid}')

!cd /content/FastSurfer && python FastSurferCNN/run_prediction.py \
    --t1 {first} \
    --asegdkt_segfile {OUT_DIR}/{sid}/mri/aparc.DKTatlas+aseg.deep.mgz \
    --conformed_name {OUT_DIR}/{sid}/mri/orig.mgz \
    --brainmask_name {OUT_DIR}/{sid}/mri/mask.mgz \
    --aseg_name {OUT_DIR}/{sid}/mri/aseg.auto_noCCseg.mgz \
    --sid {sid} --sd {OUT_DIR} \
    --device cuda --batch_size 1 --threads 4 \
    --seg_log {OUT_DIR}/{sid}/scripts/deep-seg.log \
    --vox_size min --viewagg_device auto

## 5. Full batch (~10 hrs on A100, ~20 hrs on T4)

In [ ]:
import subprocess, time
import glob

niftis = sorted(glob.glob(f'{WORK}/nifti/*.nii.gz') + glob.glob(f'{WORK}/nifti/*/*.nii.gz'))
print(f'Total scans: {len(niftis)}')

start = time.time()
ok = fail = skip = 0
for i, nifti in enumerate(niftis, 1):
    sid = os.path.basename(nifti).replace('.nii.gz', '')
    if os.path.exists(f'{OUT_DIR}/{sid}/stats/aseg+DKT.stats'):
        skip += 1
        continue
    t0 = time.time()
    r = subprocess.run([
        'python', '/content/FastSurfer/FastSurferCNN/run_prediction.py',
        '--t1', nifti,
        '--asegdkt_segfile', f'{OUT_DIR}/{sid}/mri/aparc.DKTatlas+aseg.deep.mgz',
        '--conformed_name', f'{OUT_DIR}/{sid}/mri/orig.mgz',
        '--brainmask_name', f'{OUT_DIR}/{sid}/mri/mask.mgz',
        '--aseg_name', f'{OUT_DIR}/{sid}/mri/aseg.auto_noCCseg.mgz',
        '--sid', sid, '--sd', OUT_DIR,
        '--device', 'cuda', '--batch_size', '1', '--threads', '4',
        '--seg_log', f'{OUT_DIR}/{sid}/scripts/deep-seg.log',
        '--vox_size', 'min', '--viewagg_device', 'auto',
    ], capture_output=True, text=True, timeout=1200)
    elapsed = time.time() - t0
    if os.path.exists(f'{OUT_DIR}/{sid}/stats/aseg+DKT.stats'):
        ok += 1; status = 'OK'
    else:
        fail += 1; status = 'FAIL'
    avg_min = (time.time() - start) / 60 / max(1, ok)
    eta_hrs = avg_min * (len(niftis) - i - skip) / 60 if avg_min else 0
    print(f'[{i}/{len(niftis)}] {status} {sid} ({elapsed:.0f}s)  ok={ok} fail={fail} skip={skip} avg_min={avg_min:.1f} ETA_hrs={eta_hrs:.1f}')

print(f'\nDONE  OK={ok}  SKIP={skip}  FAIL={fail}')

## 6. Package results + copy back to Drive

In [ ]:
import subprocess
# Tarball just the /stats + aparc+aseg.mgz per subject (the light, useful bits)
!cd {OUT_DIR} && find . \( -path '*/stats/*' -o -name 'aparc.DKTatlas+aseg.deep.mgz' \) -type f | tar -czf /content/fastsurfer_colab_stats.tar.gz -T -
!ls -lh /content/fastsurfer_colab_stats.tar.gz
# Copy back to Drive
!cp /content/fastsurfer_colab_stats.tar.gz {DRIVE_ROOT}/
print('Results saved to Drive:', f'{DRIVE_ROOT}/fastsurfer_colab_stats.tar.gz')